# RescueFlow AI — Autonomous Delivery Exception Resolution Agent

This notebook demonstrates **RescueFlow AI**, an autonomous delivery-operations agent for resolving delivery exceptions.

### Key Demonstrations:
1. **Multi-step Plan-Act-Observe Behavior** (calling real Python tools, inspecting results, deciding next action).
2. **Persistent Operational Memory** (past resolutions and preferences changing current decisions).
3. **Deterministic Business Logic & Hybrid AI Scoring** (Python policy engine + objective score calculation).
4. **Visible Execution Trace & Safety Guards** (error handling, verification before resolution, escalation on failure).

In [ ]:
import os
import sys

# Ensure project root is on Python path
sys.path.insert(0, os.path.abspath('..'))

from app.database.engine import init_db, get_db
from app.database.seed import seed_all
from app.database.models import Incident, Order, Customer, Delivery, AgentMemory
from app.agent.agent import AgentRunner
from app.providers.base import LLMProvider, LLMResponse
from app.services.scoring import score_option
from app.services.policy import PolicyEngine

# Reset and seed the operational database with fresh scenarios
seed_all()
print("\nDatabase initialized and seeded.")

---
## Scenario 1: Customer Unavailable → Autonomous Rescheduling

- **Incident**: Customer Aarav Sharma was not home during scheduled delivery window (09:00–11:00).
- **Memory**: Customer strongly prefers morning delivery (09:00–12:00).
- **Goal**: Investigate, discover matching morning slot tomorrow, execute reschedule, verify, and explain.

In [ ]:
with get_db() as db:
    inc1 = db.query(Incident).filter(Incident.id == 1).first()
    print(f"[Initial State] Incident #{inc1.id}: {inc1.incident_type}")
    print(f"Description: {inc1.description}")
    print(f"Initial Status: {inc1.status}")

# Define mock autonomous provider for deterministic multi-step testing
class Scenario1MockProvider(LLMProvider):
    def __init__(self):
        self.step = 0
    def generate(self, messages, tools=None, **kwargs):
        self.step += 1
        if self.step == 1:
            # Investigate order
            return LLMResponse(
                content="Investigating order 1 details.",
                tool_calls=[{"id": "t1", "type": "function", "function": {"name": "get_order", "arguments": {"order_id": 1}}}]
            )
        elif self.step == 2:
            # Investigate customer and delivery history
            return LLMResponse(
                content="Order verified. Now loading customer profile and delivery attempts.",
                tool_calls=[
                    {"id": "t2a", "type": "function", "function": {"name": "get_customer_profile", "arguments": {"customer_id": 1}}},
                    {"id": "t2b", "type": "function", "function": {"name": "get_delivery_history", "arguments": {"order_id": 1}}}
                ]
            )
        elif self.step == 3:
            # Find available delivery slots
            return LLMResponse(
                content="Customer prefers morning. Checking available slots for tomorrow (2026-09-14).",
                tool_calls=[{"id": "t3", "type": "function", "function": {"name": "find_available_delivery_slots", "arguments": {"customer_id": 1, "date": "2026-09-14"}}}]
            )
        elif self.step == 4:
            # Reschedule delivery
            return LLMResponse(
                content="Found morning slot 09:00-12:00 matching customer preference. Executing reschedule.",
                tool_calls=[{"id": "t4", "type": "function", "function": {"name": "reschedule_delivery", "arguments": {"order_id": 1, "date": "2026-09-14", "start_time": "09:00", "end_time": "12:00"}}}]
            )
        else:
            # Final business decision
            return LLMResponse(
                content="Delivery rescheduled successfully for tomorrow (2026-09-14) 09:00–12:00. The slot matches the customer's morning preference and has confirmed courier availability."
            )

runner1 = AgentRunner(provider=Scenario1MockProvider())
result1 = runner1.resolve_incident(1)

print("\n--- AGENT EXECUTION TRACE ---")
for event in result1.trace:
    if event.event_type == "tool_call":
        print(f"Step {event.step_number}: Tool '{event.tool_name}' -> Result: {event.result.get('success')}")
    elif event.event_type == "decision":
        print(f"Final Decision: {event.summary}")

print(f"\nFinal Result Status: {result1.status}")
print(f"Decision Summary: {result1.decision_summary}")

with get_db() as db:
    deliv1 = db.query(Delivery).filter(Delivery.order_id == 1).first()
    print(f"[Verified DB State] Scheduled: {deliv1.scheduled_date} {deliv1.scheduled_start}-{deliv1.scheduled_end} | Status: {deliv1.status}")

---
## Scenario 2: Preferred Slot Unavailable → Agent Finds Feasible Alternative

- **Incident**: Incident #2 for Priya Patel (courier delayed, original window missed).
- **Preference**: Customer prefers morning delivery.
- **Constraint**: Tomorrow morning couriers are unavailable in Mumbai, but afternoon slot 13:00–16:00 is available.
- **Agent Behavior**: Discovers unavailability through tools, selects afternoon slot, and explicitly explains trade-off.

In [ ]:
with get_db() as db:
    inc2 = db.query(Incident).filter(Incident.id == 2).first()
    print(f"[Initial State] Incident #{inc2.id}: {inc2.incident_type}")
    print(f"Description: {inc2.description}")

class Scenario2MockProvider(LLMProvider):
    def __init__(self):
        self.step = 0
    def generate(self, messages, tools=None, **kwargs):
        self.step += 1
        if self.step == 1:
            return LLMResponse(
                content="Investigating incident #2 by checking order and slots.",
                tool_calls=[{"id": "t1", "type": "function", "function": {"name": "get_order", "arguments": {"order_id": 2}}}]
            )
        elif self.step == 2:
            return LLMResponse(
                content="Checking available delivery slots.",
                tool_calls=[{"id": "t2", "type": "function", "function": {"name": "find_available_delivery_slots", "arguments": {"customer_id": 2, "date": "2026-09-14"}}}]
            )
        elif self.step == 3:
            # Discovered only afternoon slot is feasible
            return LLMResponse(
                content="Morning slot unavailable. Selecting next best feasible afternoon slot (13:00-16:00).",
                tool_calls=[{"id": "t3", "type": "function", "function": {"name": "reschedule_delivery", "arguments": {"order_id": 2, "date": "2026-09-14", "start_time": "13:00", "end_time": "16:00"}}}]
            )
        else:
            return LLMResponse(
                content="Delivery rescheduled for 2026-09-14 13:00–16:00. Note: Customer prefers morning, but morning slots were unavailable due to courier capacity. Afternoon slot selected as the next best operational alternative to avoid a 48-hour delay."
            )

runner2 = AgentRunner(provider=Scenario2MockProvider())
result2 = runner2.resolve_incident(2)

print("\n--- AGENT EXECUTION TRACE ---")
for event in result2.trace:
    if event.event_type == "tool_call":
        print(f"Step {event.step_number}: Tool '{event.tool_name}' -> Result: {event.result.get('success')}")
    elif event.event_type == "decision":
        print(f"Final Decision: {event.summary}")

print(f"\nFinal Result Status: {result2.status}")
print(f"Trade-off Explanation: {result2.decision_summary}")

---
## Scenario 3: Repeated Failures → Memory Retrieval & Autonomous Escalation

- **Incident**: Incident #4 for Rohan Gupta (repeated delivery failure).
- **History**: 3 consecutive failed delivery attempts on 2026-09-11, 2026-09-12, and 2026-09-13.
- **Memory**: Customer has travel history and requested guard handoff after previous escalation.
- **Agent Behavior**: Discovers repeated failures, evaluates policy rule #4, and autonomously escalates rather than blindly rescheduling.

In [ ]:
with get_db() as db:
    inc4 = db.query(Incident).filter(Incident.id == 4).first()
    print(f"[Initial State] Incident #{inc4.id}: {inc4.incident_type} (Severity: {inc4.severity})")
    print(f"Description: {inc4.description}")

class Scenario3MockProvider(LLMProvider):
    def __init__(self):
        self.step = 0
    def generate(self, messages, tools=None, **kwargs):
        self.step += 1
        if self.step == 1:
            return LLMResponse(
                content="Investigating repeated failure on order 4.",
                tool_calls=[{"id": "t1", "type": "function", "function": {"name": "get_order", "arguments": {"order_id": 4}}}]
            )
        elif self.step == 2:
            return LLMResponse(
                content="Checking delivery history to verify attempt count and prior notes.",
                tool_calls=[{"id": "t2", "type": "function", "function": {"name": "get_delivery_history", "arguments": {"order_id": 4}}}]
            )
        elif self.step == 3:
            # 3 failed attempts confirmed; escalate
            return LLMResponse(
                content="Three failed attempts confirmed. Blind rescheduling violates Policy Rule #4. Escalating incident to logistics supervisor.",
                tool_calls=[{"id": "t3", "type": "function", "function": {
                    "name": "escalate_incident",
                    "arguments": {
                        "order_id": 4,
                        "reason": "Repeated customer unavailability (3 attempts). Customer memory indicates travel schedule. Escalating to human ops for guard handoff authorization."
                    }
                }}]
            )
        else:
            return LLMResponse(
                content="Incident escalated to Human Logistics Operations. Reason: 3 consecutive delivery attempts failed. Persistent memory indicates customer travels frequently. Blind rescheduling halted; security guard handoff or alternative address requested from customer."
            )

runner3 = AgentRunner(provider=Scenario3MockProvider())
result3 = runner3.resolve_incident(4)

print("\n--- AGENT EXECUTION TRACE ---")
for event in result3.trace:
    if event.event_type == "tool_call":
        print(f"Step {event.step_number}: Tool '{event.tool_name}' -> Result: {event.result.get('success')}")
    elif event.event_type == "decision":
        print(f"Final Decision: {event.summary}")

print(f"\nFinal Result Status: {result3.status}")
print(f"Escalation Summary: {result3.decision_summary}")

with get_db() as db:
    inc_check = db.query(Incident).filter(Incident.id == 4).first()
    mem_check = db.query(AgentMemory).filter(AgentMemory.customer_id == 3).order_by(AgentMemory.id.desc()).first()
    print(f"[Verified DB Incident Status]: {inc_check.status}")
    print(f"[New Memory Persisted]: [{mem_check.memory_type}] {mem_check.content}")